# 🤖 AI Engineering Fundamentals — Lezione 4
## Notebook Gruppo A

**ITS Novitas 4.0 | Giovedì 28/05/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [ ]:
GRUPPO = "A"
MEMBRI = ["", "", "", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

In [ ]:
# ⚠️ Prima esecuzione: ChromaDB scarica Sentence Transformers (~90MB)
# Avviate questa cella per prima e aspettate il completamento
!pip install anthropic chromadb pypdf sentence-transformers -q

from google.colab import userdata
import anthropic, os, chromadb
from pypdf import PdfReader

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
chroma_client = chromadb.Client()

def chiedi_claude(messaggio, system=None, max_tokens=600):
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": messaggio}]
    }
    if system:
        params["system"] = system
    return client.messages.create(**params).content[0].text

print("✅ Setup completato!")

In [ ]:
# Documento WiData di test — usatelo per tutti gli esercizi
DOCUMENTO_WIDATA = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica
e qualità dell'aria (CO2, PM2.5). Classificazione IP67: impermeabile e resistente alla polvere.
Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni. Connettività: LoRaWAN, NB-IoT, WiFi.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare. Temperatura operativa: -40°C a +70°C.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato).

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
"""

with open("manuale_widata.txt", "w", encoding="utf-8") as f:
    f.write(DOCUMENTO_WIDATA)

print(f"✅ Documento creato: {len(DOCUMENTO_WIDATA)} caratteri")

---
## 🎯 Tema del Gruppo A: Pipeline RAG completa

Costruite e testate l'intera pipeline RAG — dall'indicizzazione
del documento alla risposta finale, confrontando con e senza RAG.

---
### Esercizio 1 — Indicizzare il documento *(guidato)*

Spezzate il documento in chunk e indicizzatelo in ChromaDB.
Verificate che l'indicizzazione sia andata a buon fine.

In [ ]:
# Esercizio 1 — indicizzazione

def chunka_testo(testo, chunk_size=400, overlap=50):
    """Spezza il testo in chunk con overlap."""
    chunks = []
    start = 0
    while start < len(testo):
        chunk = testo[start:start+chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# Chunka il documento
chunks = chunka_testo(DOCUMENTO_WIDATA)
print(f"Chunk generati: {len(chunks)}")
print(f"\nPrimo chunk ({len(chunks[0])} char):")
print(chunks[0])
print(f"\nSecondo chunk ({len(chunks[1])} char):")
print(chunks[1])
print()
print("💡 Notate l'overlap: le ultime parole del chunk 1 appaiono all'inizio del chunk 2?")

In [ ]:
# Indicizza in ChromaDB
# TODO: create una collection chiamata 'widata_gruppo_a'
# e aggiungete i chunk
collection = chroma_client.get_or_create_collection(___)
collection.add(
    documents=___,
    ids=[str(i) for i in range(len(chunks))]
)

print(f"✅ Indicizzati {collection.count()} chunk in ChromaDB")

---
### Esercizio 2 — RAG completo: con vs senza *(guidato)*

La demo più importante della lezione.
Stessa domanda con e senza RAG — la differenza deve essere netta.

In [ ]:
# Esercizio 2 — con vs senza RAG

SYSTEM_WIDATA = """
Sei l'assistente virtuale di WiData Srl.
Rispondi SOLO basandoti sui documenti forniti nel contesto.
Se la risposta non è nei documenti, dì chiaramente:
'Non ho questa informazione nei miei documenti.'
Non inventare mai dati tecnici, prezzi o specifiche.
"""

def cerca(domanda, n=3):
    """Recupera i chunk più rilevanti per la domanda."""
    # TODO: usate collection.query() con query_texts e n_results
    risultati = collection.query(
        query_texts=___,
        n_results=___
    )
    return risultati["documents"][0]

def chat_rag(domanda):
    """Chatbot con RAG: recupera contesto e genera risposta."""
    chunks_trovati = cerca(domanda)
    contesto = "\n\n---\n\n".join(chunks_trovati)
    prompt = f"""Documenti di riferimento:

{contesto}

---

Domanda: {domanda}"""
    return chiedi_claude(prompt, system=SYSTEM_WIDATA)

# Test con 3 domande
domande = [
    "Qual è l'autonomia della batteria del sensore XS200?",
    "Quanti sensori gestisce il gateway GW500?",
    "Quanto costa il piano Pro di Xplore?",
]

for domanda in domande:
    print(f"\n{'='*55}")
    print(f"❓ {domanda}")
    print()

    # TODO: chiamate chiedi_claude SENZA RAG (solo system prompt)
    print("🔴 SENZA RAG:")
    print(___)

    # TODO: chiamate chat_rag CON RAG
    print("\n🟢 CON RAG:")
    print(___)

---
### Esercizio 3 — Visualizzare il retrieval *(libero)*

Stampate i chunk recuperati per ogni domanda.
Sono rilevanti? ChromaDB ha trovato le informazioni giuste?
Provate anche domande fuori dal documento — cosa succede?

In [ ]:
# Esercizio 3 — visualizzare il retrieval

def chat_rag_debug(domanda):
    """Come chat_rag ma stampa anche i chunk recuperati."""
    chunks_trovati = cerca(domanda)

    print(f"📄 Chunk recuperati ({len(chunks_trovati)}):")
    for i, chunk in enumerate(chunks_trovati):
        print(f"  Chunk {i+1}: {chunk[:100]}...")
    print()

    contesto = "\n\n---\n\n".join(chunks_trovati)
    prompt = f"Documenti:\n\n{contesto}\n\n---\n\nDomanda: {domanda}"
    risposta = chiedi_claude(prompt, system=SYSTEM_WIDATA)
    print(f"🤖 Risposta: {risposta}")
    return risposta

# Testate con domande diverse — dentro e fuori dal documento
domande_test = [
    "Quali certificazioni ha il sensore XS200?",
    "Come si contatta il supporto tecnico?",
    "Offrite sensori subacquei?",           # non nel documento
    "Qual è il prezzo del modello Enterprise?",  # non specificato
]

for domanda in domande_test:
    print(f"\n{'='*55}")
    print(f"❓ {domanda}")
    chat_rag_debug(domanda)

# Osservazione: il sistema si comporta correttamente
# quando la risposta non è nel documento?
# ...

---
### Esercizio 4 — RAG + conversation history *(libero)*

Integrate RAG con la conversation history della Lezione 3.
Il chatbot deve usare sia il contesto RAG che la memoria
della conversazione precedente.

In [ ]:
# Esercizio 4 — RAG + history

history = []

def chat_rag_con_storia(domanda):
    """Chatbot con RAG + conversation history."""
    # 1. Recupera chunk rilevanti
    chunks_trovati = cerca(domanda)
    contesto = "\n\n---\n\n".join(chunks_trovati)

    # 2. Costruisci il messaggio con contesto RAG
    messaggio_con_rag = f"""Documenti di riferimento:

{contesto}

---

Domanda: {domanda}"""

    # TODO: aggiungete il messaggio alla history
    # chiamate l'API con tutta la history e system=SYSTEM_WIDATA
    # aggiungete la risposta alla history
    # restituite la risposta
    ___

# Test: domande collegate che richiedono sia RAG che memoria
print(chat_rag_con_storia("Parlami del sensore XS200."))
print(chat_rag_con_storia("Qual è la sua autonomia?"))      # usa RAG
print(chat_rag_con_storia("Di quale sensore stavamo parlando?"))  # usa history

# Il chatbot risponde correttamente a tutte e tre?
# ...

---
## 📊 Preparate la presentazione (5 slide)

1. **Il problema** — demo senza RAG: cosa inventa Claude su WiData?
2. **La pipeline** — le due fasi spiegate con il vostro documento
3. **Con vs senza RAG** — i vostri risultati a confronto
4. **Il retrieval** — mostrate i chunk recuperati per una domanda
5. **RAG + history** — come funzionano insieme e quando serve ciascuno

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*